In [0]:
# ============================================================================
# GETYOURGUIDE PRICING FEATURES AUDIT
# Comprehensive Analysis of Supplier Pricing Configurations
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from decimal import Decimal

plt.style.use('default')
sns.set_palette("husl")

# ============================================================================
# SECTION 1: MARKETPLACE OVERVIEW
# ============================================================================

print("="*100)
print("SECTION 1: MARKETPLACE OVERVIEW")
print("="*100)

overview_df = spark.sql("""
SELECT 
  COUNT(DISTINCT tour_id) as total_tours,
  COUNT(DISTINCT supplier_id) as total_suppliers,
  SUM(num_tour_options) as total_tour_options,
  ROUND(AVG(num_tour_options), 1) as avg_options_per_tour,
  COUNT(DISTINCT location_name) as total_locations,
  COUNT(DISTINCT activity_type) as total_activity_types
FROM production.supply_analytics.tour_feature_summary
""").toPandas()

print("\nMarketplace Size:")
display(overview_df)

# Chart 1: Tours by Location
location_df = spark.sql("""
SELECT 
  location_name,
  COUNT(DISTINCT tour_id) as num_tours
FROM production.supply_analytics.tour_feature_summary
GROUP BY location_name
ORDER BY num_tours DESC
LIMIT 20
""").toPandas()

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(len(location_df)), location_df['num_tours'], color='#3498db')
ax.set_yticks(range(len(location_df)))
ax.set_yticklabels(location_df['location_name'])
ax.set_xlabel('Number of Tours', fontsize=13, fontweight='bold')
ax.set_title('Top 20 Locations by Tour Count', fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(location_df['num_tours']):
    ax.text(v + 10, i, f'{v:,}', va='center', fontweight='bold')
plt.tight_layout()
display(fig)
plt.close()

# Chart 2: Tours by Activity Type
activity_df = spark.sql("""
SELECT 
  activity_type,
  COUNT(DISTINCT tour_id) as num_tours
FROM production.supply_analytics.tour_feature_summary
GROUP BY activity_type
ORDER BY num_tours DESC
LIMIT 15
""").toPandas()

fig, ax = plt.subplots(figsize=(12, 7))
ax.bar(range(len(activity_df)), activity_df['num_tours'], color='#e74c3c')
ax.set_xticks(range(len(activity_df)))
ax.set_xticklabels(activity_df['activity_type'], rotation=45, ha='right')
ax.set_ylabel('Number of Tours', fontsize=13, fontweight='bold')
ax.set_title('Tours by Activity Type', fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# SECTION 2: FEATURE ADOPTION OVERVIEW
# ============================================================================

print("\n" + "="*100)
print("SECTION 2: FEATURE ADOPTION OVERVIEW")
print("="*100)

adoption_df = spark.sql("""
SELECT 
  SUM(CAST(has_individual_pricing AS INT)) as tours_with_individual,
  SUM(CAST(has_group_pricing AS INT)) as tours_with_group,
  SUM(CAST(has_addons_configured AS INT)) as tours_with_addons,
  SUM(CAST(has_any_scale_pricing AS INT)) as tours_with_scale_pricing,
  SUM(CAST(has_capacity_limits AS INT)) as tours_with_capacity,
  SUM(CAST(has_cutoff_configured AS INT)) as tours_with_cutoff,
  COUNT(*) as total_tours,
  
  ROUND(100.0 * SUM(CAST(has_individual_pricing AS INT)) / COUNT(*), 1) as individual_pct,
  ROUND(100.0 * SUM(CAST(has_group_pricing AS INT)) / COUNT(*), 1) as group_pct,
  ROUND(100.0 * SUM(CAST(has_addons_configured AS INT)) / COUNT(*), 1) as addon_pct,
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1) as scale_pct,
  ROUND(100.0 * SUM(CAST(has_capacity_limits AS INT)) / COUNT(*), 1) as capacity_pct,
  ROUND(100.0 * SUM(CAST(has_cutoff_configured AS INT)) / COUNT(*), 1) as cutoff_pct
FROM production.supply_analytics.tour_feature_summary
WHERE uses_external_pricing = 0
""").toPandas()

print("\nFeature Adoption (Native Pricing Tours Only):")
display(adoption_df)

# Chart 3: Feature Adoption Rates
fig, ax = plt.subplots(figsize=(12, 7))
features = ['Individual\nPricing', 'Group\nPricing', 'Addons', 'Scale\nPricing', 'Capacity\nLimits', 'Cutoff\nTime']
percentages = [
    adoption_df['individual_pct'].values[0],
    adoption_df['group_pct'].values[0],
    adoption_df['addon_pct'].values[0],
    adoption_df['scale_pct'].values[0],
    adoption_df['capacity_pct'].values[0],
    adoption_df['cutoff_pct'].values[0]
]
colors = ['#3498db', '#9b59b6', '#f39c12', '#2ecc71', '#e74c3c', '#1abc9c']

bars = ax.bar(range(len(features)), percentages, color=colors, edgecolor='black', linewidth=1.5)
ax.set_xticks(range(len(features)))
ax.set_xticklabels(features, fontsize=12)
ax.set_ylabel('Adoption Rate (%)', fontsize=13, fontweight='bold')
ax.set_title('Pricing Feature Adoption Rates', fontsize=15, fontweight='bold', pad=20)
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)

for i, (bar, pct) in enumerate(zip(bars, percentages)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# SECTION 3: INDIVIDUAL PRICING DEEP DIVE
# ============================================================================

print("\n" + "="*100)
print("SECTION 3: INDIVIDUAL PRICING ANALYSIS")
print("="*100)

print("\n3.1 Individual Pricing Overview:")
individual_overview = spark.sql("""
SELECT 
  COUNT(DISTINCT tour_id) as tours_with_individual,
  ROUND(AVG(avg_num_individual_categories), 2) as avg_categories_per_tour,
  ROUND(AVG(avg_individual_price_tiers), 1) as avg_tiers_per_tour,
  SUM(CAST(has_scale_pricing_individual AS INT)) as tours_with_scale_pricing
FROM production.supply_analytics.tour_feature_summary
WHERE has_individual_pricing = 1
  AND uses_external_pricing = 0
""").toPandas()

display(individual_overview)

print("\n3.2 Most Common Individual Pricing Categories:")
individual_categories = spark.sql("""
SELECT 
  ticket_category,
  min_age,
  max_age,
  booking_type,
  num_tours_using,
  total_occurrences,
  avg_tiers_per_category
FROM production.supply_analytics.individual_category_analysis
ORDER BY num_tours_using DESC
LIMIT 20
""").toPandas()

display(individual_categories)

# Chart 4: Top Individual Categories
fig, ax = plt.subplots(figsize=(14, 9))
top_cats = individual_categories.head(15)
category_labels = [f"{row['ticket_category']}\n({row['min_age']}-{row['max_age']} yrs)" 
                   for _, row in top_cats.iterrows()]

ax.barh(range(len(top_cats)), top_cats['num_tours_using'], color='#3498db', edgecolor='black', linewidth=1.2)
ax.set_yticks(range(len(top_cats)))
ax.set_yticklabels(category_labels, fontsize=11)
ax.set_xlabel('Number of Tours Using Category', fontsize=13, fontweight='bold')
ax.set_title('Top 15 Individual Pricing Categories', fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

for i, v in enumerate(top_cats['num_tours_using']):
    ax.text(v + 50, i, f'{v:,}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
display(fig)
plt.close()

print("\n3.3 Individual Pricing Tier Structures:")
individual_tiers = spark.sql("""
SELECT 
  ticket_category,
  max_participants,
  num_tours,
  median_retail_price,
  avg_retail_price
FROM production.supply_analytics.individual_pricing_tier_patterns
WHERE num_tours >= 100
ORDER BY ticket_category, max_participants
LIMIT 50
""").toPandas()

display(individual_tiers)

# Chart 5: Pricing Tiers by Category
fig, ax = plt.subplots(figsize=(14, 8))
categories = individual_tiers['ticket_category'].unique()[:5]
for cat in categories:
    cat_data = individual_tiers[individual_tiers['ticket_category'] == cat]
    ax.plot(cat_data['max_participants'], cat_data['median_retail_price'], 
            marker='o', linewidth=2.5, markersize=8, label=cat)

ax.set_xlabel('Maximum Participants (Tier Threshold)', fontsize=13, fontweight='bold')
ax.set_ylabel('Median Retail Price (EUR)', fontsize=13, fontweight='bold')
ax.set_title('Pricing Tier Structure by Category Type', fontsize=15, fontweight='bold', pad=20)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# SECTION 4: GROUP PRICING DEEP DIVE
# ============================================================================

print("\n" + "="*100)
print("SECTION 4: GROUP PRICING ANALYSIS")
print("="*100)

print("\n4.1 Group Pricing Overview:")
group_overview = spark.sql("""
SELECT 
  COUNT(DISTINCT tour_id) as tours_with_group,
  ROUND(AVG(avg_num_group_categories), 2) as avg_categories_per_tour,
  ROUND(AVG(avg_group_price_tiers), 1) as avg_tiers_per_tour,
  SUM(CAST(has_scale_pricing_group AS INT)) as tours_with_scale_pricing
FROM production.supply_analytics.tour_feature_summary
WHERE has_group_pricing = 1
  AND uses_external_pricing = 0
""").toPandas()

display(group_overview)

print("\n4.2 Group Pricing Patterns:")
group_patterns = spark.sql("""
SELECT 
  is_additional_pax,
  num_price_tiers,
  num_tours_using,
  total_occurrences
FROM production.supply_analytics.group_pricing_analysis
ORDER BY num_tours_using DESC
""").toPandas()

display(group_patterns)

print("\n4.3 Group Pricing Tier Structures:")
group_tiers = spark.sql("""
SELECT 
  is_additional_pax,
  max_participants,
  num_tours,
  median_retail_price,
  avg_retail_price
FROM production.supply_analytics.group_pricing_tier_patterns
WHERE num_tours >= 50
ORDER BY is_additional_pax, max_participants
""").toPandas()

display(group_tiers)

# Chart 6: Group Pricing Tiers
fig, ax = plt.subplots(figsize=(12, 7))
for is_additional in group_tiers['is_additional_pax'].unique():
    data = group_tiers[group_tiers['is_additional_pax'] == is_additional]
    label = 'Additional Pax' if is_additional else 'Base Group'
    ax.plot(data['max_participants'], data['median_retail_price'], 
            marker='o', linewidth=2.5, markersize=8, label=label)

ax.set_xlabel('Maximum Participants', fontsize=13, fontweight='bold')
ax.set_ylabel('Median Retail Price (EUR)', fontsize=13, fontweight='bold')
ax.set_title('Group Pricing Tier Structure', fontsize=15, fontweight='bold', pad=20)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# SECTION 5: ADDON ANALYSIS
# ============================================================================

print("\n" + "="*100)
print("SECTION 5: ADDON ANALYSIS")
print("="*100)

print("\n5.1 Addon Overview:")
addon_overview = spark.sql("""
SELECT 
  COUNT(DISTINCT tour_id) as tours_with_addons,
  ROUND(AVG(avg_num_addons), 2) as avg_addons_per_tour,
  ROUND(AVG(avg_addon_price_tiers), 1) as avg_tiers_per_addon,
  SUM(CAST(has_scale_pricing_addons AS INT)) as tours_with_addon_scale_pricing
FROM production.supply_analytics.tour_feature_summary
WHERE has_addons_configured = 1
  AND uses_external_pricing = 0
""").toPandas()

display(addon_overview)

print("\n5.2 Most Used Addons:")
addon_usage = spark.sql("""
SELECT 
  addon_id,
  num_price_tiers,
  num_tours_using,
  avg_tiers_per_addon
FROM production.supply_analytics.addon_analysis
ORDER BY num_tours_using DESC
LIMIT 30
""").toPandas()

display(addon_usage)

# Chart 7: Top Addons by Usage
fig, ax = plt.subplots(figsize=(14, 8))
top_addons = addon_usage.head(20)
ax.bar(range(len(top_addons)), top_addons['num_tours_using'], color='#9b59b6', edgecolor='black', linewidth=1.2)
ax.set_xticks(range(len(top_addons)))
ax.set_xticklabels(top_addons['addon_id'], rotation=45, ha='right')
ax.set_ylabel('Number of Tours', fontsize=13, fontweight='bold')
ax.set_xlabel('Addon ID', fontsize=13, fontweight='bold')
ax.set_title('Top 20 Addons by Tour Usage', fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

print("\n5.3 Addon Pricing Tier Patterns:")
addon_tiers = spark.sql("""
SELECT 
  max_participants,
  num_tours,
  num_unique_addons,
  median_retail_price,
  avg_retail_price
FROM production.supply_analytics.addon_pricing_tier_patterns
ORDER BY max_participants
""").toPandas()

display(addon_tiers)

# ============================================================================
# SECTION 6: SCALE PRICING ANALYSIS
# ============================================================================

print("\n" + "="*100)
print("SECTION 6: SCALE PRICING (TIERED PRICING) ANALYSIS")
print("="*100)

scale_summary = spark.sql("""
SELECT 
  SUM(CAST(has_any_scale_pricing AS INT)) as tours_with_any_scale,
  SUM(CAST(has_scale_pricing_individual AS INT)) as tours_scale_individual,
  SUM(CAST(has_scale_pricing_group AS INT)) as tours_scale_group,
  SUM(CAST(has_scale_pricing_addons AS INT)) as tours_scale_addons,
  
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1) as any_scale_pct,
  ROUND(100.0 * SUM(CAST(has_scale_pricing_individual AS INT)) / COUNT(*), 1) as individual_scale_pct,
  ROUND(100.0 * SUM(CAST(has_scale_pricing_group AS INT)) / COUNT(*), 1) as group_scale_pct,
  ROUND(100.0 * SUM(CAST(has_scale_pricing_addons AS INT)) / COUNT(*), 1) as addon_scale_pct
  
FROM production.supply_analytics.tour_feature_summary
WHERE uses_external_pricing = 0
""").toPandas()

display(scale_summary)

# Chart 8: Scale Pricing Adoption Breakdown
fig, ax = plt.subplots(figsize=(10, 7))
scale_types = ['Any Scale\nPricing', 'Individual\nScale', 'Group\nScale', 'Addon\nScale']
scale_pcts = [
    scale_summary['any_scale_pct'].values[0],
    scale_summary['individual_scale_pct'].values[0],
    scale_summary['group_scale_pct'].values[0],
    scale_summary['addon_scale_pct'].values[0]
]
colors_scale = ['#2ecc71', '#3498db', '#9b59b6', '#f39c12']

bars = ax.bar(range(len(scale_types)), scale_pcts, color=colors_scale, edgecolor='black', linewidth=1.5)
ax.set_xticks(range(len(scale_types)))
ax.set_xticklabels(scale_types, fontsize=12)
ax.set_ylabel('Adoption Rate (%)', fontsize=13, fontweight='bold')
ax.set_title('Scale Pricing Adoption by Type', fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

for i, (bar, pct) in enumerate(zip(bars, scale_pcts)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# SECTION 7: OPERATIONAL FEATURES
# ============================================================================

print("\n" + "="*100)
print("SECTION 7: OPERATIONAL FEATURES (CAPACITY & CUTOFF)")
print("="*100)

operational_summary = spark.sql("""
SELECT 
  SUM(CAST(has_capacity_limits AS INT)) as tours_with_capacity,
  SUM(CAST(has_cutoff_configured AS INT)) as tours_with_cutoff,
  SUM(CAST(has_min_participant_requirement AS INT)) as tours_with_min_pax,
  SUM(CAST(has_max_participant_limit AS INT)) as tours_with_max_pax,
  
  ROUND(AVG(avg_dates_covered), 0) as avg_date_coverage_days,
  ROUND(AVG(avg_pct_vacancy_coverage), 1) as avg_vacancy_coverage_pct
  
FROM production.supply_analytics.tour_feature_summary
WHERE uses_external_pricing = 0
""").toPandas()

display(operational_summary)

# ============================================================================
# SECTION 8: FEATURE COMBINATIONS & SOPHISTICATION
# ============================================================================

print("\n" + "="*100)
print("SECTION 8: FEATURE SOPHISTICATION ANALYSIS")
print("="*100)

print("\n8.1 Feature Count Distribution:")
feature_sophistication = spark.sql("""
SELECT 
  tour_feature_count,
  COUNT(DISTINCT tour_id) as num_tours,
  ROUND(100.0 * COUNT(DISTINCT tour_id) / SUM(COUNT(DISTINCT tour_id)) OVER (), 1) as pct_of_tours
FROM production.supply_analytics.tour_feature_summary
WHERE uses_external_pricing = 0
GROUP BY tour_feature_count
ORDER BY tour_feature_count
""").toPandas()

display(feature_sophistication)

# Chart 9: Feature Sophistication Distribution
fig, ax = plt.subplots(figsize=(12, 7))
ax.bar(feature_sophistication['tour_feature_count'], 
       feature_sophistication['num_tours'], 
       color='#e74c3c', edgecolor='black', linewidth=1.5)
ax.set_xlabel('Number of Advanced Features (0-5)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Tours', fontsize=13, fontweight='bold')
ax.set_title('Tour Distribution by Feature Sophistication', fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

for i, row in feature_sophistication.iterrows():
    ax.text(row['tour_feature_count'], row['num_tours'] + 500,
            f"{row['pct_of_tours']:.1f}%", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

print("\n8.2 Most Common Category Combinations:")
category_combos = spark.sql("""
SELECT 
  num_categories_in_combo,
  COUNT(*) as num_combination_patterns,
  SUM(num_tours_using) as total_tour_occurrences
FROM production.supply_analytics.category_combination_patterns
GROUP BY num_categories_in_combo
ORDER BY num_categories_in_combo
""").toPandas()

display(category_combos)

# ============================================================================
# SECTION 9: GEOGRAPHIC & ACTIVITY BREAKDOWN
# ============================================================================

print("\n" + "="*100)
print("SECTION 9: FEATURE ADOPTION BY LOCATION & ACTIVITY")
print("="*100)

print("\n9.1 Feature Adoption by Top 20 Locations:")
location_features = spark.sql("""
SELECT 
  location_name,
  COUNT(DISTINCT tour_id) as total_tours,
  ROUND(100.0 * SUM(CAST(has_individual_pricing AS INT)) / COUNT(*), 1) as individual_pct,
  ROUND(100.0 * SUM(CAST(has_group_pricing AS INT)) / COUNT(*), 1) as group_pct,
  ROUND(100.0 * SUM(CAST(has_addons_configured AS INT)) / COUNT(*), 1) as addon_pct,
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1) as scale_pct,
  ROUND(100.0 * SUM(CAST(has_capacity_limits AS INT)) / COUNT(*), 1) as capacity_pct
FROM production.supply_analytics.tour_feature_summary
WHERE uses_external_pricing = 0
GROUP BY location_name
HAVING COUNT(DISTINCT tour_id) >= 100
ORDER BY total_tours DESC
LIMIT 20
""").toPandas()

display(location_features)

# Chart 10: Feature Adoption Heatmap by Location
fig, ax = plt.subplots(figsize=(16, 10))
heatmap_data = location_features[['individual_pct', 'group_pct', 'addon_pct', 
                                   'scale_pct', 'capacity_pct']].values.T

im = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(len(location_features)))
ax.set_yticks(range(5))
ax.set_xticklabels(location_features['location_name'], rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(['Individual', 'Group', 'Addons', 'Scale', 'Capacity'], fontsize=12, fontweight='bold')

for i in range(5):
    for j in range(len(location_features)):
        ax.text(j, i, f'{heatmap_data[i, j]:.0f}%', ha="center", va="center", 
                color="black", fontsize=9, fontweight='bold')

ax.set_title('Feature Adoption by Top 20 Locations (%)', fontsize=16, fontweight='bold', pad=20)
cbar = fig.colorbar(im, ax=ax, label='Adoption Rate (%)')
cbar.set_label('Adoption Rate (%)', fontsize=12, fontweight='bold')
plt.tight_layout()
display(fig)
plt.close()

print("\n9.2 Feature Adoption by Activity Type:")
activity_features = spark.sql("""
SELECT 
  activity_type,
  COUNT(DISTINCT tour_id) as total_tours,
  ROUND(100.0 * SUM(CAST(has_individual_pricing AS INT)) / COUNT(*), 1) as individual_pct,
  ROUND(100.0 * SUM(CAST(has_group_pricing AS INT)) / COUNT(*), 1) as group_pct,
  ROUND(100.0 * SUM(CAST(has_addons_configured AS INT)) / COUNT(*), 1) as addon_pct,
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1) as scale_pct
FROM production.supply_analytics.tour_feature_summary
WHERE uses_external_pricing = 0
GROUP BY activity_type
HAVING COUNT(DISTINCT tour_id) >= 50
ORDER BY total_tours DESC
LIMIT 15
""").toPandas()

display(activity_features)

# ============================================================================
# SECTION 10: API INTEGRATION & EXTERNAL PRICING
# ============================================================================

print("\n" + "="*100)
print("SECTION 10: API INTEGRATION & EXTERNAL PRICING")
print("="*100)

api_summary = spark.sql("""
SELECT 
  SUM(CAST(uses_prices_over_api AS INT)) as uses_prices_api,
  SUM(CAST(uses_live_availability_and_price AS INT)) as uses_live_api,
  SUM(CAST(uses_pricing_scales_over_api AS INT)) as uses_scales_api,
  SUM(CAST(uses_external_pricing AS INT)) as uses_any_external,
  COUNT(*) as total_tours,
  
  ROUND(100.0 * SUM(CAST(uses_external_pricing AS INT)) / COUNT(*), 1) as external_pct
FROM production.supply_analytics.tour_feature_summary
""").toPandas()

display(api_summary)

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*100)
print("AUDIT COMPLETE - EXECUTIVE SUMMARY")
print("="*100)

final_summary = spark.sql("""
SELECT 
  'Total Tours' as metric,
  COUNT(DISTINCT tour_id) as value
FROM production.supply_analytics.tour_feature_summary

UNION ALL
SELECT 'Tours with Individual Pricing', SUM(CAST(has_individual_pricing AS INT))
FROM production.supply_analytics.tour_feature_summary

UNION ALL
SELECT 'Tours with Group Pricing', SUM(CAST(has_group_pricing AS INT))
FROM production.supply_analytics.tour_feature_summary

UNION ALL
SELECT 'Tours with Addons', SUM(CAST(has_addons_configured AS INT))
FROM production.supply_analytics.tour_feature_summary

UNION ALL
SELECT 'Tours with Scale Pricing', SUM(CAST(has_any_scale_pricing AS INT))
FROM production.supply_analytics.tour_feature_summary

UNION ALL
SELECT 'Tours with Capacity Limits', SUM(CAST(has_capacity_limits AS INT))
FROM production.supply_analytics.tour_feature_summary

UNION ALL
SELECT 'Tours with Cutoff Time', SUM(CAST(has_cutoff_configured AS INT))
FROM production.supply_analytics.tour_feature_summary

UNION ALL
SELECT 'Tours Using External Pricing', SUM(CAST(uses_external_pricing AS INT))
FROM production.supply_analytics.tour_feature_summary
""").toPandas()

display(final_summary)

print("\n✅ PRICING FEATURES AUDIT COMPLETE")
print("="*100)


In [0]:
# ============================================================================
# GETYOURGUIDE PRICING FEATURES AUDIT
# Complete Analysis of Supplier Pricing Configurations
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')
sns.set_palette("husl")

# ============================================================================
# EXECUTIVE SUMMARY - HIGH LEVEL OVERVIEW
# ============================================================================

print("="*100)
print(" " * 30 + "GETYOURGUIDE PRICING FEATURES AUDIT")
print(" " * 35 + "EXECUTIVE SUMMARY")
print("="*100)

overview_query = """
SELECT 
  'MARKETPLACE OVERVIEW' as section,
  'Total Active Tours' as metric,
  COUNT(DISTINCT tour_id) as value,
  NULL as pct_of_tours
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'MARKETPLACE OVERVIEW', 'Total Active Suppliers', COUNT(DISTINCT supplier_id), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'MARKETPLACE OVERVIEW', 'Total Tour Options', SUM(num_tour_options), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'MARKETPLACE OVERVIEW', 'Avg Tour Options per Tour', ROUND(AVG(num_tour_options), 1), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'INDIVIDUAL PRICING', 'Tours with Individual Pricing', SUM(CAST(has_individual_pricing AS INT)),
ROUND(100.0 * SUM(CAST(has_individual_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'INDIVIDUAL PRICING', 'Avg Individual Categories per Tour', ROUND(AVG(avg_num_individual_categories), 2), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'INDIVIDUAL PRICING', 'Avg Individual Price Tiers per Tour', ROUND(AVG(avg_individual_price_tiers), 1), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'INDIVIDUAL PRICING', 'Tours with Scale Pricing on Individual', SUM(CAST(has_scale_pricing_individual AS INT)),
ROUND(100.0 * SUM(CAST(has_scale_pricing_individual AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'GROUP PRICING', 'Tours with Group Pricing', SUM(CAST(has_group_pricing AS INT)),
ROUND(100.0 * SUM(CAST(has_group_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'GROUP PRICING', 'Avg Group Categories per Tour', ROUND(AVG(avg_num_group_categories), 2), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'GROUP PRICING', 'Avg Group Price Tiers per Tour', ROUND(AVG(avg_group_price_tiers), 1), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'GROUP PRICING', 'Tours with Scale Pricing on Group', SUM(CAST(has_scale_pricing_group AS INT)),
ROUND(100.0 * SUM(CAST(has_scale_pricing_group AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'ADDONS', 'Tours with Addons Configured', SUM(CAST(has_addons_configured AS INT)),
ROUND(100.0 * SUM(CAST(has_addons_configured AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'ADDONS', 'Avg Number of Addons per Tour', ROUND(AVG(avg_num_addons), 2), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'ADDONS', 'Avg Addon Price Tiers per Tour', ROUND(AVG(avg_addon_price_tiers), 1), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'ADDONS', 'Tours with Scale Pricing on Addons', SUM(CAST(has_scale_pricing_addons AS INT)),
ROUND(100.0 * SUM(CAST(has_scale_pricing_addons AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'SCALE PRICING', 'Tours with Any Scale Pricing', SUM(CAST(has_any_scale_pricing AS INT)),
ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'SCALE PRICING', 'Avg Total Price Tiers per Tour', ROUND(AVG(avg_total_price_tiers), 1), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'API INTEGRATION', 'Tours Using Prices Over API', SUM(CAST(uses_prices_over_api AS INT)),
ROUND(100.0 * SUM(CAST(uses_prices_over_api AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'API INTEGRATION', 'Tours Using Live Availability and Price', SUM(CAST(uses_live_availability_and_price AS INT)),
ROUND(100.0 * SUM(CAST(uses_live_availability_and_price AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'API INTEGRATION', 'Tours Using Pricing Scales Over API', SUM(CAST(uses_pricing_scales_over_api AS INT)),
ROUND(100.0 * SUM(CAST(uses_pricing_scales_over_api AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'API INTEGRATION', 'Tours Using Any External Pricing', SUM(CAST(uses_external_pricing AS INT)),
ROUND(100.0 * SUM(CAST(uses_external_pricing AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'OPERATIONAL FEATURES', 'Tours with Cutoff Time Configured', SUM(CAST(has_cutoff_configured AS INT)),
ROUND(100.0 * SUM(CAST(has_cutoff_configured AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'OPERATIONAL FEATURES', 'Tours with Capacity Limits', SUM(CAST(has_capacity_limits AS INT)),
ROUND(100.0 * SUM(CAST(has_capacity_limits AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'OPERATIONAL FEATURES', 'Tours with Min Participant Requirement', SUM(CAST(has_min_participant_requirement AS INT)),
ROUND(100.0 * SUM(CAST(has_min_participant_requirement AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'OPERATIONAL FEATURES', 'Tours with Max Participant Limit', SUM(CAST(has_max_participant_limit AS INT)),
ROUND(100.0 * SUM(CAST(has_max_participant_limit AS INT)) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'OPERATIONAL HEALTH', 'Avg Date Coverage (Days in Next 12 Months)', ROUND(AVG(avg_dates_covered), 0), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'OPERATIONAL HEALTH', 'Avg Vacancy Coverage (% of Dates)', ROUND(AVG(avg_pct_vacancy_coverage), 1), NULL
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'FEATURE SOPHISTICATION', 'Tours with 0 Advanced Features', SUM(CASE WHEN tour_feature_count = 0 THEN 1 ELSE 0 END),
ROUND(100.0 * SUM(CASE WHEN tour_feature_count = 0 THEN 1 ELSE 0 END) / COUNT(*), 1)
FROM production.supply_analytics.tour_feature_audit

UNION ALL SELECT 'FEATURE SOPHISTICATION', 'Avg Feature Count per Tour (0-5)', ROUND(AVG(tour_feature_count), 2), NULL
FROM production.supply_analytics.tour_feature_audit

ORDER BY section, metric
"""

overview_df = spark.sql(overview_query).toPandas()
display(overview_df)

# Visualization: Feature Adoption Summary
feature_adoption = overview_df[overview_df['pct_of_tours'].notna() & (overview_df['section'].isin(['INDIVIDUAL PRICING', 'GROUP PRICING', 'ADDONS', 'SCALE PRICING', 'OPERATIONAL FEATURES']))]

fig, ax = plt.subplots(figsize=(14, 8))
sections = feature_adoption['section'].unique()
colors_map = {'INDIVIDUAL PRICING': '#3498db', 'GROUP PRICING': '#9b59b6', 'ADDONS': '#f39c12', 
              'SCALE PRICING': '#2ecc71', 'OPERATIONAL FEATURES': '#e74c3c'}

x_pos = 0
for section in sections:
    section_data = feature_adoption[feature_adoption['section'] == section]
    num_metrics = len(section_data)
    positions = range(x_pos, x_pos + num_metrics)
    
    ax.bar(positions, section_data['pct_of_tours'], 
           color=colors_map.get(section, '#95a5a6'), 
           label=section, alpha=0.85, edgecolor='black', linewidth=1.2)
    
    for i, (pos, pct) in enumerate(zip(positions, section_data['pct_of_tours'])):
        ax.text(pos, pct + 2, f'{pct:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    x_pos += num_metrics + 1

ax.set_ylabel('Adoption Rate (%)', fontsize=13, fontweight='bold')
ax.set_title('Pricing Feature Adoption Rates Across Marketplace', fontsize=15, fontweight='bold', pad=20)
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
display(fig)
plt.close()

# ============================================================================
# PART 1: INDIVIDUAL PRICING DEEP DIVE
# ============================================================================

print("\n" + "="*100)
print(" " * 35 + "PART 1: INDIVIDUAL PRICING")
print("="*100)

print("\n1.1 MOST COMMON INDIVIDUAL PRICING CATEGORIES")
print("-" * 100)

individual_categories = spark.sql("""
SELECT 
  ticket_category,
  CONCAT(COALESCE(min_age, 0), '-', COALESCE(max_age, 999)) as age_range,
  booking_type,
  num_tours_using,
  total_occurrences,
  ROUND(avg_tiers_per_category, 1) as avg_tiers
FROM production.supply_analytics.individual_category_analysis
ORDER BY num_tours_using DESC
LIMIT 30
""").toPandas()

display(individual_categories)

# Chart: Top Categories
fig, ax = plt.subplots(figsize=(14, 9))
top_20 = individual_categories.head(20)
category_labels = [f"{row['ticket_category']} ({row['age_range']})" for _, row in top_20.iterrows()]

bars = ax.barh(range(len(top_20)), top_20['num_tours_using'], color='#3498db', edgecolor='black', linewidth=1.2)
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(category_labels, fontsize=10)
ax.set_xlabel('Number of Tours', fontsize=13, fontweight='bold')
ax.set_title('Top 20 Individual Pricing Categories by Tour Usage', fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

for i, v in enumerate(top_20['num_tours_using']):
    ax.text(v + 50, i, f'{v:,}', va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
display(fig)
plt.close()

print("\n1.2 INDIVIDUAL PRICING TIER PATTERNS")
print("-" * 100)

individual_tiers = spark.sql("""
SELECT 
  ticket_category,
  max_participants as tier_threshold,
  num_tours,
  ROUND(avg_retail_price, 2) as avg_price_eur,
  ROUND(median_retail_price, 2) as median_price_eur,
  ROUND(p25_retail_price, 2) as p25_price,
  ROUND(p75_retail_price, 2) as p75_price
FROM production.supply_analytics.individual_pricing_tier_patterns
WHERE num_tours >= 100
ORDER BY ticket_category, max_participants
LIMIT 100
""").toPandas()

display(individual_tiers)

# Chart: Price by Tier for Top Categories
fig, ax = plt.subplots(figsize=(14, 8))
top_categories = individual_tiers['ticket_category'].unique()[:5]

for cat in top_categories:
    cat_data = individual_tiers[individual_tiers['ticket_category'] == cat]
    ax.plot(cat_data['tier_threshold'], cat_data['median_price_eur'], 
            marker='o', linewidth=2.5, markersize=8, label=cat)

ax.set_xlabel('Tier Threshold (Max Participants)', fontsize=13, fontweight='bold')
ax.set_ylabel('Median Price (EUR)', fontsize=13, fontweight='bold')
ax.set_title('Individual Pricing Tier Structure by Category', fontsize=15, fontweight='bold', pad=20)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

print("\n1.3 INDIVIDUAL PRICING BY LOCATION")
print("-" * 100)

individual_by_location = spark.sql("""
WITH category_location AS (
  SELECT 
    fab.location_name,
    ind.ticketCategory as ticket_category,
    COUNT(DISTINCT fab.tour_id) as num_tours,
    ROUND(AVG(prices.retailPrice), 2) as avg_price_eur
  FROM production.supply_analytics.feature_audit_base fab
  LATERAL VIEW EXPLODE(individual_categories) exploded_ind AS ind
  LATERAL VIEW EXPLODE(ind.prices) exploded_prices AS prices
  WHERE fab.has_individual_pricing = 1
    AND fab.uses_external_pricing = 0
  GROUP BY fab.location_name, ind.ticketCategory
  HAVING COUNT(DISTINCT fab.tour_id) >= 50
)
SELECT 
  location_name,
  ticket_category,
  num_tours,
  avg_price_eur
FROM category_location
ORDER BY location_name, num_tours DESC
""").toPandas()

display(individual_by_location.head(50))

print("\n1.4 INDIVIDUAL PRICING BY ACTIVITY TYPE")
print("-" * 100)

individual_by_activity = spark.sql("""
WITH category_activity AS (
  SELECT 
    fab.activity_type,
    ind.ticketCategory as ticket_category,
    COUNT(DISTINCT fab.tour_id) as num_tours,
    ROUND(AVG(prices.retailPrice), 2) as avg_price_eur,
    ROUND(PERCENTILE(prices.retailPrice, 0.50), 2) as median_price_eur
  FROM production.supply_analytics.feature_audit_base fab
  LATERAL VIEW EXPLODE(individual_categories) exploded_ind AS ind
  LATERAL VIEW EXPLODE(ind.prices) exploded_prices AS prices
  WHERE fab.has_individual_pricing = 1
    AND fab.uses_external_pricing = 0
  GROUP BY fab.activity_type, ind.ticketCategory
  HAVING COUNT(DISTINCT fab.tour_id) >= 30
)
SELECT 
  activity_type,
  ticket_category,
  num_tours,
  avg_price_eur,
  median_price_eur
FROM category_activity
ORDER BY activity_type, num_tours DESC
""").toPandas()

display(individual_by_activity.head(50))

# ============================================================================
# PART 2: GROUP PRICING DEEP DIVE
# ============================================================================

print("\n" + "="*100)
print(" " * 35 + "PART 2: GROUP PRICING")
print("="*100)

print("\n2.1 GROUP PRICING ADOPTION PATTERNS")
print("-" * 100)

group_patterns = spark.sql("""
SELECT 
  is_additional_pax,
  num_price_tiers,
  num_tours_using,
  total_occurrences
FROM production.supply_analytics.group_pricing_analysis
ORDER BY num_tours_using DESC
""").toPandas()

display(group_patterns)

print("\n2.2 GROUP PRICING TIER STRUCTURE")
print("-" * 100)

group_tiers = spark.sql("""
SELECT 
  is_additional_pax,
  max_participants as tier_threshold,
  num_tours,
  ROUND(avg_retail_price, 2) as avg_price_eur,
  ROUND(median_retail_price, 2) as median_price_eur,
  ROUND(p25_retail_price, 2) as p25_price,
  ROUND(p75_retail_price, 2) as p75_price
FROM production.supply_analytics.group_pricing_tier_patterns
WHERE num_tours >= 50
ORDER BY is_additional_pax, max_participants
""").toPandas()

display(group_tiers)

# Chart: Group Pricing Tiers
fig, ax = plt.subplots(figsize=(12, 7))
for is_additional in group_tiers['is_additional_pax'].unique():
    data = group_tiers[group_tiers['is_additional_pax'] == is_additional]
    label = 'Additional Pax Pricing' if is_additional else 'Base Group Pricing'
    ax.plot(data['tier_threshold'], data['median_price_eur'], 
            marker='o', linewidth=3, markersize=10, label=label)

ax.set_xlabel('Group Size Threshold (Max Participants)', fontsize=13, fontweight='bold')
ax.set_ylabel('Median Price (EUR)', fontsize=13, fontweight='bold')
ax.set_title('Group Pricing Tier Structure', fontsize=15, fontweight='bold', pad=20)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

print("\n2.3 GROUP PRICING BY LOCATION")
print("-" * 100)

group_by_location = spark.sql("""
WITH group_location AS (
  SELECT 
    fab.location_name,
    COUNT(DISTINCT fab.tour_id) as num_tours,
    ROUND(AVG(prices.retailPrice), 2) as avg_group_price_eur,
    ROUND(PERCENTILE(prices.retailPrice, 0.50), 2) as median_group_price_eur
  FROM production.supply_analytics.feature_audit_base fab
  LATERAL VIEW EXPLODE(group_categories) exploded_grp AS grp
  LATERAL VIEW EXPLODE(grp.prices) exploded_prices AS prices
  WHERE fab.has_group_pricing = 1
    AND fab.uses_external_pricing = 0
  GROUP BY fab.location_name
  HAVING COUNT(DISTINCT fab.tour_id) >= 50
)
SELECT *
FROM group_location
ORDER BY num_tours DESC
""").toPandas()

display(group_by_location)

print("\n2.4 GROUP PRICING BY ACTIVITY TYPE")
print("-" * 100)

group_by_activity = spark.sql("""
WITH group_activity AS (
  SELECT 
    fab.activity_type,
    COUNT(DISTINCT fab.tour_id) as num_tours,
    ROUND(AVG(prices.retailPrice), 2) as avg_group_price_eur,
    ROUND(PERCENTILE(prices.retailPrice, 0.50), 2) as median_group_price_eur,
    ROUND(AVG(grp.prices[0].maxParticipants), 1) as avg_group_size_tier_1
  FROM production.supply_analytics.feature_audit_base fab
  LATERAL VIEW EXPLODE(group_categories) exploded_grp AS grp
  LATERAL VIEW EXPLODE(grp.prices) exploded_prices AS prices
  WHERE fab.has_group_pricing = 1
    AND fab.uses_external_pricing = 0
  GROUP BY fab.activity_type
  HAVING COUNT(DISTINCT fab.tour_id) >= 30
)
SELECT *
FROM group_activity
ORDER BY num_tours DESC
""").toPandas()

display(group_by_activity)

# ============================================================================
# PART 3: ADDON PRICING DEEP DIVE
# ============================================================================

print("\n" + "="*100)
print(" " * 35 + "PART 3: ADDON ANALYSIS")
print("="*100)

print("\n3.1 MOST COMMON ADDONS")
print("-" * 100)

addon_usage = spark.sql("""
SELECT 
  addon_id,
  num_price_tiers,
  num_tours_using,
  total_occurrences,
  ROUND(avg_tiers_per_addon, 1) as avg_tiers
FROM production.supply_analytics.addon_analysis
ORDER BY num_tours_using DESC
LIMIT 50
""").toPandas()

display(addon_usage)

# Chart: Top Addons
fig, ax = plt.subplots(figsize=(14, 8))
top_30_addons = addon_usage.head(30)
ax.bar(range(len(top_30_addons)), top_30_addons['num_tours_using'], 
       color='#9b59b6', edgecolor='black', linewidth=1.2)
ax.set_xticks(range(len(top_30_addons)))
ax.set_xticklabels(top_30_addons['addon_id'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Number of Tours', fontsize=13, fontweight='bold')
ax.set_xlabel('Addon ID', fontsize=13, fontweight='bold')
ax.set_title('Top 30 Addons by Tour Usage', fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

print("\n3.2 ADDON PRICING TIER STRUCTURE")
print("-" * 100)

addon_tiers = spark.sql("""
SELECT 
  max_participants as tier_threshold,
  num_tours,
  num_unique_addons,
  ROUND(avg_retail_price, 2) as avg_price_eur,
  ROUND(median_retail_price, 2) as median_price_eur,
  ROUND(p25_retail_price, 2) as p25_price,
  ROUND(p75_retail_price, 2) as p75_price
FROM production.supply_analytics.addon_pricing_tier_patterns
ORDER BY max_participants
""").toPandas()

display(addon_tiers)

# Chart: Addon Pricing Distribution
fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(addon_tiers['tier_threshold'], addon_tiers['median_price_eur'], 
        marker='o', linewidth=3, markersize=10, color='#9b59b6', label='Median Price')
ax.fill_between(addon_tiers['tier_threshold'], 
                addon_tiers['p25_price'], 
                addon_tiers['p75_price'], 
                alpha=0.3, color='#9b59b6', label='25th-75th Percentile')

ax.set_xlabel('Participant Tier Threshold', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (EUR)', fontsize=13, fontweight='bold')
ax.set_title('Addon Pricing Distribution by Tier', fontsize=15, fontweight='bold', pad=20)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

print("\n3.3 ADDON PRICING BY LOCATION")
print("-" * 100)

addon_by_location = spark.sql("""
WITH addon_location AS (
  SELECT 
    fab.location_name,
    addon.addonId as addon_id,
    COUNT(DISTINCT fab.tour_id) as num_tours,
    ROUND(AVG(prices.retailPrice), 2) as avg_addon_price_eur
  FROM production.supply_analytics.feature_audit_base fab
  LATERAL VIEW EXPLODE(addons) exploded_addon AS addon
  LATERAL VIEW EXPLODE(addon.prices) exploded_prices AS prices
  WHERE fab.has_addons_configured = 1
    AND fab.uses_external_pricing = 0
  GROUP BY fab.location_name, addon.addonId
  HAVING COUNT(DISTINCT fab.tour_id) >= 20
)
SELECT 
  location_name,
  COUNT(DISTINCT addon_id) as num_unique_addons,
  SUM(num_tours) as total_addon_tour_combinations,
  ROUND(AVG(avg_addon_price_eur), 2) as avg_addon_price_location
FROM addon_location
GROUP BY location_name
ORDER BY num_unique_addons DESC
""").toPandas()

display(addon_by_location)

print("\n3.4 ADDON PRICING BY ACTIVITY TYPE")
print("-" * 100)

addon_by_activity = spark.sql("""
WITH addon_activity AS (
  SELECT 
    fab.activity_type,
    addon.addonId as addon_id,
    COUNT(DISTINCT fab.tour_id) as num_tours,
    ROUND(AVG(prices.retailPrice), 2) as avg_addon_price_eur
  FROM production.supply_analytics.feature_audit_base fab
  LATERAL VIEW EXPLODE(addons) exploded_addon AS addon
  LATERAL VIEW EXPLODE(addon.prices) exploded_prices AS prices
  WHERE fab.has_addons_configured = 1
    AND fab.uses_external_pricing = 0
  GROUP BY fab.activity_type, addon.addonId
  HAVING COUNT(DISTINCT fab.tour_id) >= 15
)
SELECT 
  activity_type,
  COUNT(DISTINCT addon_id) as num_unique_addons,
  SUM(num_tours) as total_addon_tour_combinations,
  ROUND(AVG(avg_addon_price_eur), 2) as avg_addon_price_activity
FROM addon_activity
GROUP BY activity_type
ORDER BY num_unique_addons DESC
""").toPandas()

display(addon_by_activity)

# ============================================================================
# PART 4: SCALE PRICING (TIERED PRICING) DEEP DIVE
# ============================================================================

print("\n" + "="*100)
print(" " * 35 + "PART 4: SCALE PRICING ANALYSIS")
print("="*100)

print("\n4.1 SCALE PRICING ADOPTION BY TYPE")
print("-" * 100)

scale_adoption = spark.sql("""
SELECT 
  SUM(CAST(has_any_scale_pricing AS INT)) as tours_any_scale,
  SUM(CAST(has_scale_pricing_individual AS INT)) as tours_scale_individual,
  SUM(CAST(has_scale_pricing_group AS INT)) as tours_scale_group,
  SUM(CAST(has_scale_pricing_addons AS INT)) as tours_scale_addons,
  
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1) as any_scale_pct,
  ROUND(100.0 * SUM(CAST(has_scale_pricing_individual AS INT)) / COUNT(*), 1) as individual_scale_pct,
  ROUND(100.0 * SUM(CAST(has_scale_pricing_group AS INT)) / COUNT(*), 1) as group_scale_pct,
  ROUND(100.0 * SUM(CAST(has_scale_pricing_addons AS INT)) / COUNT(*), 1) as addon_scale_pct,
  
  ROUND(AVG(avg_total_price_tiers), 1) as avg_total_tiers
  
FROM production.supply_analytics.tour_feature_audit
WHERE uses_external_pricing = 0
""").toPandas()

display(scale_adoption)

print("\n4.2 MOST COMMON TIER THRESHOLDS - INDIVIDUAL PRICING")
print("-" * 100)

tier_thresholds_individual = spark.sql("""
SELECT 
  max_participants as tier_threshold,
  COUNT(DISTINCT tour_id) as num_tours_using_threshold,
  COUNT(*) as total_occurrences,
  ROUND(AVG(median_retail_price), 2) as avg_price_at_threshold
FROM production.supply_analytics.individual_pricing_tier_patterns
WHERE num_tours >= 50
GROUP BY max_participants
ORDER BY num_tours_using_threshold DESC
LIMIT 20
""").toPandas()

display(tier_thresholds_individual)

# Chart: Common Tier Thresholds
fig, ax = plt.subplots(figsize=(12, 7))
ax.bar(tier_thresholds_individual['tier_threshold'].astype(str), 
       tier_thresholds_individual['num_tours_using_threshold'], 
       color='#2ecc71', edgecolor='black', linewidth=1.2)
ax.set_xlabel('Participant Tier Threshold', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Tours Using Threshold', fontsize=13, fontweight='bold')
ax.set_title('Most Common Scale Pricing Tier Thresholds (Individual)', fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
display(fig)
plt.close()

print("\n4.3 SCALE PRICING ADOPTION BY LOCATION")
print("-" * 100)

scale_by_location = spark.sql("""
SELECT 
  location_name,
  COUNT(DISTINCT tour_id) as total_tours,
  SUM(CAST(has_any_scale_pricing AS INT)) as tours_with_scale,
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1) as scale_adoption_pct,
  ROUND(AVG(avg_total_price_tiers), 1) as avg_tiers_per_tour
FROM production.supply_analytics.tour_feature_audit
WHERE uses_external_pricing = 0
GROUP BY location_name
HAVING COUNT(DISTINCT tour_id) >= 100
ORDER BY tours_with_scale DESC
LIMIT 30
""").toPandas()

display(scale_by_location)

print("\n4.4 SCALE PRICING ADOPTION BY ACTIVITY TYPE")
print("-" * 100)

scale_by_activity = spark.sql("""
SELECT 
  activity_type,
  COUNT(DISTINCT tour_id) as total_tours,
  SUM(CAST(has_any_scale_pricing AS INT)) as tours_with_scale,
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1) as scale_adoption_pct,
  ROUND(AVG(avg_total_price_tiers), 1) as avg_tiers_per_tour
FROM production.supply_analytics.tour_feature_audit
WHERE uses_external_pricing = 0
GROUP BY activity_type
HAVING COUNT(DISTINCT tour_id) >= 50
ORDER BY tours_with_scale DESC
""").toPandas()

display(scale_by_activity)

# ============================================================================
# PART 5: GEOGRAPHIC & ACTIVITY HEATMAPS
# ============================================================================

print("\n" + "="*100)
print(" " * 30 + "PART 5: GEOGRAPHIC & ACTIVITY BREAKDOWN")
print("="*100)

print("\n5.1 FEATURE ADOPTION HEATMAP BY LOCATION")
print("-" * 100)

location_heatmap = spark.sql("""
SELECT 
  location_name,
  COUNT(DISTINCT tour_id) as total_tours,
  ROUND(100.0 * SUM(CAST(has_individual_pricing AS INT)) / COUNT(*), 1) as individual_pct,
  ROUND(100.0 * SUM(CAST(has_group_pricing AS INT)) / COUNT(*), 1) as group_pct,
  ROUND(100.0 * SUM(CAST(has_addons_configured AS INT)) / COUNT(*), 1) as addon_pct,
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1) as scale_pct,
  ROUND(100.0 * SUM(CAST(has_capacity_limits AS INT)) / COUNT(*), 1) as capacity_pct,
  ROUND(100.0 * SUM(CAST(has_cutoff_configured AS INT)) / COUNT(*), 1) as cutoff_pct
FROM production.supply_analytics.tour_feature_audit
WHERE uses_external_pricing = 0
GROUP BY location_name
HAVING COUNT(DISTINCT tour_id) >= 100
ORDER BY total_tours DESC
LIMIT 25
""").toPandas()

display(location_heatmap)

# Heatmap Visualization
fig, ax = plt.subplots(figsize=(16, 12))
heatmap_data = location_heatmap[['individual_pct', 'group_pct', 'addon_pct', 
                                   'scale_pct', 'capacity_pct', 'cutoff_pct']].values.T

im = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(len(location_heatmap)))
ax.set_yticks(range(6))
ax.set_xticklabels(location_heatmap['location_name'], rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(['Individual', 'Group', 'Addons', 'Scale', 'Capacity', 'Cutoff'], fontsize=12, fontweight='bold')

for i in range(6):
    for j in range(len(location_heatmap)):
        ax.text(j, i, f'{heatmap_data[i, j]:.0f}%', ha="center", va="center", 
                color="black", fontsize=8, fontweight='bold')

ax.set_title('Feature Adoption Heatmap by Top 25 Locations (%)', fontsize=16, fontweight='bold', pad=20)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Adoption Rate (%)', fontsize=12, fontweight='bold')
plt.tight_layout()
display(fig)
plt.close()

print("\n5.2 AVERAGE PRICING BY LOCATION & CATEGORY")
print("-" * 100)

pricing_by_location = spark.sql("""
WITH location_pricing AS (
  SELECT 
    fab.location_name,
    ind.ticketCategory,
    ROUND(AVG(prices.retailPrice), 2) as avg_price_eur,
    COUNT(DISTINCT fab.tour_id) as num_tours
  FROM production.supply_analytics.feature_audit_base fab
  LATERAL VIEW EXPLODE(individual_categories) exploded_ind AS ind
  LATERAL VIEW EXPLODE(ind.prices) exploded_prices AS prices
  WHERE fab.uses_external_pricing = 0
  GROUP BY fab.location_name, ind.ticketCategory
  HAVING COUNT(DISTINCT fab.tour_id) >= 30
)
SELECT *
FROM location_pricing
ORDER BY location_name, num_tours DESC
""").toPandas()

display(pricing_by_location.head(100))

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*100)
print(" " * 30 + "AUDIT COMPLETE - KEY TAKEAWAYS")
print("="*100)

key_metrics = spark.sql("""
SELECT 
  COUNT(DISTINCT tour_id) as total_tours,
  COUNT(DISTINCT supplier_id) as total_suppliers,
  ROUND(100.0 * SUM(CAST(has_individual_pricing AS INT)) / COUNT(*), 1) as individual_adoption,
  ROUND(100.0 * SUM(CAST(has_group_pricing AS INT)) / COUNT(*), 1) as group_adoption,
  ROUND(100.0 * SUM(CAST(has_addons_configured AS INT)) / COUNT(*), 1) as addon_adoption,
  ROUND(100.0 * SUM(CAST(has_any_scale_pricing AS INT)) / COUNT(*), 1) as scale_adoption,
  ROUND(AVG(tour_feature_count), 2) as avg_features_per_tour
FROM production.supply_analytics.tour_feature_audit
WHERE uses_external_pricing = 0
""").toPandas()

display(key_metrics)

print("\n✅ COMPREHENSIVE PRICING FEATURES AUDIT COMPLETE")
print("="*100)
